# 03 — Accessibilité au bus de chaque carreau (objectif 1)

**Rôle dans le pipeline :** pour chaque carreau habité, mesurer l'arrêt le plus proche et l'offre de bus à moins de 5 minutes à pied, puis classer le carreau dans un niveau d'offre.

| Lit | Produit |
|---|---|
| `silver/carreaux.parquet` + `silver/offre_arrets.parquet` | `silver/carreaux_acces.parquet` |

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from parametres import (SILVER, COMMUNES, RAYON_TERRE_M, RAYON_5MIN_M, SEUIL_ISOLE_M,
                        SEUILS_OFFRE, NIVEAUX_OFFRE)

carreaux = pd.read_parquet(SILVER / "carreaux.parquet")
arrets = pd.read_parquet(SILVER / "offre_arrets.parquet")
print(len(carreaux), "carreaux |", len(arrets), "arrêts |", arrets["reseau"].unique())

3691 carreaux | 2583 arrêts | ["Kar'Ouest" 'Car Jaune']


### Fonction `acces_bus`

- **`BallTree`** range les arrêts pour retrouver très vite les plus proches de chaque carreau, au lieu de comparer chaque carreau à chaque arrêt.
- **`metric="haversine"`** : distance sur la sphère terrestre, à partir de coordonnées en **radians**. Le résultat est un angle, multiplié par le rayon de la Terre pour obtenir des mètres.
- **`passages_5min`** additionne les passages de tous les arrêts à moins de 5 min. Un même bus est compté plusieurs fois s'il dessert plusieurs arrêts proches : c'est un **indice d'offre**, pas un nombre exact de bus.

**Limites :** distance **à vol d'oiseau** (la marche réelle est plus longue, surtout dans les Hauts : on **surestime** l'accès) ; position au **centre** du carreau (jusqu'à ~140 m d'écart).

In [2]:
def acces_bus(carreaux, arrets):
    arbre = BallTree(np.radians(arrets[["stop_lat", "stop_lon"]].values), metric="haversine")
    points = np.radians(carreaux[["lat", "lon"]].values)
    dist, _ = arbre.query(points, k=1)
    voisins = arbre.query_radius(points, r=RAYON_5MIN_M / RAYON_TERRE_M)
    passages = arrets["nb_passages"].values
    return pd.DataFrame({
        "dist_arret_m": dist[:, 0] * RAYON_TERRE_M,
        "nb_arrets_5min": [len(v) for v in voisins],
        "passages_5min": [passages[v].sum() for v in voisins],
    }, index=carreaux.index)


def niveau_offre(passages):
    return pd.cut(passages, bins=SEUILS_OFFRE, labels=NIVEAUX_OFFRE)

### Tous réseaux confondus

Pas besoin de filtrer les arrêts Car Jaune de l'Ouest : la recherche « à moins de 400 m d'un carreau » ignore d'elle-même les arrêts de Saint-Denis ou Saint-Pierre.

Valeurs attendues avec **Kar'Ouest seul** (lundi, Fabric) : **95,4 %** des habitants à moins de 400 m d'un arrêt, **9 986** sans arrêt, distance médiane **181 m**, maximum **5 945 m**.

In [3]:
carreaux = carreaux.join(acces_bus(carreaux, arrets))
carreaux["niveau_offre"] = niveau_offre(carreaux["passages_5min"]).astype(str)

pop = carreaux["ind"].sum()
pop_desservie = carreaux.loc[carreaux["nb_arrets_5min"] > 0, "ind"].sum()
print(f"Habitants avec un arrêt à moins de {RAYON_5MIN_M} m : {round(pop_desservie)} ({pop_desservie / pop:.1%})")
print("Sans arrêt :", round(pop - pop_desservie))
carreaux[["dist_arret_m", "nb_arrets_5min", "passages_5min"]].describe().round(0)

Habitants avec un arrêt à moins de 400 m : 209678 (95.6%)
Sans arrêt : 9728


,dist_arret_m,nb_arrets_5min,passages_5min
count,3691.0,3691.0,3691.0
mean,259.0,6.0,169.0
std,355.0,5.0,213.0
min,2.0,0.0,0.0
25%,106.0,2.0,33.0
50%,181.0,6.0,105.0
75%,300.0,10.0,220.0
max,5945.0,48.0,2193.0


### Habitants par niveau d'offre : chaque réseau seul, puis tous ensemble

Les seuils restent les mêmes dans toutes les colonnes, sinon la comparaison n'aurait pas de sens. Le passage d'une colonne à l'autre montre **ce qu'apporte chaque réseau**.

In [4]:
comparaison = {}
for reseau, arrets_reseau in arrets.groupby("reseau"):
    acces = acces_bus(carreaux, arrets_reseau)
    comparaison[f"{reseau} seul"] = carreaux.groupby(niveau_offre(acces["passages_5min"]), observed=False)["ind"].sum()
comparaison["Tous réseaux"] = carreaux.groupby("niveau_offre")["ind"].sum().reindex(NIVEAUX_OFFRE)

tableau = pd.DataFrame(comparaison).round()
tableau["% (tous réseaux)"] = (100 * tableau["Tous réseaux"] / pop).round(1)
tableau

,Car Jaune seul,Kar'Ouest seul,Tous réseaux,% (tous réseaux)
Aucun arrêt,183124.0,9986.0,9728.0,4.4
Très faible,2802.0,7714.0,7452.0,3.4
Faible,22615.0,39114.0,34173.0,15.6
Correcte,9901.0,56902.0,59109.0,26.9
Forte,963.0,105690.0,108944.0,49.7


### Carreaux isolés (à plus de 2 km de tout arrêt)

Un bus roule sur une route : un carreau très loin de tout arrêt est **suspect** (pas de route ?). Constat de lundi : **27 carreaux, 118 habitants**, en deux groupes — **Mafate** (aucune route, à exclure à terme) et les **Hauts de Saint-Leu** (routes, donc à garder : cibles d'équité).

Pour l'instant on **n'exclut rien** : la vraie règle viendra des données routières (OpenStreetMap), en enrichissement.

In [5]:
isoles = carreaux[carreaux["dist_arret_m"] > SEUIL_ISOLE_M]
print(f"Carreaux à plus de {SEUIL_ISOLE_M} m d'un arrêt : {len(isoles)} | habitants : {round(isoles['ind'].sum())}")
isoles.groupby(isoles["commune"].map(COMMUNES)).agg(carreaux=("ind", "size"), habitants=("ind", "sum")).round()

Carreaux à plus de 2000 m d'un arrêt : 27 | habitants : 118


,carreaux,habitants
commune,,
La Possession,11,65.0
Saint-Leu,11,26.0
Saint-Paul,2,14.0
Trois-Bassins,3,12.0


In [6]:
carreaux.to_parquet(SILVER / "carreaux_acces.parquet", index=False)
print("Écrit :", SILVER / "carreaux_acces.parquet")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\data\silver\carreaux_acces.parquet
